In [ ]:
# Parameters cell — Papermill overrides these at runtime
dataset_size = 10
start_base = 2440000
size_increment = 1000
contig = '2RL'
sample_set = "1354-VO-KE-DONNELLY-VMF00281"

In [ ]:
import malariagen_data

af1 = malariagen_data.Af1()

In [ ]:
import csv
import subprocess
import time
from pathlib import Path
import filelock
from memory_profiler import memory_usage
from collections.abc import Callable


def measure_regions(start_base: int, size_increment:int, dataset_size:int, sample_set:str,
                    contig: str, output_csv_prefix:str="memory_benchmarks"):
    sample_id = str(
        af1.sample_metadata(sample_sets=sample_set).iloc[0]["sample_id"]
    )

    def measure_and_write(output_path: Path, region:str, operation: Callable):
        start_time = time.perf_counter()
        peak_memory_mib = memory_usage((operation, ()), max_usage=True)
        runtime_sec = time.perf_counter() - start_time

        record = {
            "dataset_size": dataset_size,
            "size_increment": size_increment,
            "start_base": start_base,
            "sample_set": sample_set,
            "sample_id": sample_id,
            "region": region,
            "peak_memory_mib": round(peak_memory_mib, 2),
            "runtime_sec": round(runtime_sec, 4),
        }

        lock = filelock.FileLock(f"{output_path}.lock")
        with lock:
            file_exists = output_path.exists() and output_path.stat().st_size > 0
            fieldnames = list(record)
            if file_exists:
                with output_path.open("r", newline="") as f:
                    reader = csv.DictReader(f)
                    existing_fieldnames = reader.fieldnames or []
                    existing_records = list(reader)
                if any(name not in fieldnames for name in existing_fieldnames):
                    raise ValueError(
                        f"CSV {output_path} has columns not present in the current record: "
                        f"{existing_fieldnames}"
                    )
                if existing_fieldnames != fieldnames:
                    with output_path.open("w", newline="") as f:
                        writer = csv.DictWriter(f, fieldnames=fieldnames)
                        writer.writeheader()
                        writer.writerows(existing_records)
            with output_path.open("a", newline="") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                if not file_exists:
                    writer.writeheader()
                writer.writerow(record)

        print(f"Recorded {region} to {output_path}: {record}")

    for i in range(dataset_size):
        region_start = start_base + i * size_increment
        region = f"{contig}:{region_start}-{region_start + size_increment}"

        def view_region():
            af1.view_alignments(
                sample=sample_id,
                region=region,
            )

        measure_and_write(Path(output_csv_prefix, ".csv"), region, view_region)
        

In [ ]:
measure_regions(
    start_base=start_base,
    size_increment=size_increment,
    dataset_size=dataset_size,
    sample_set=sample_set,
    contig=contig,
    output_csv_prefix=output_csv_prefix,
)